## Notebook Workflow Structure

This notebook systematically tests Epic medical history annotations extraction and retrieval functionality:

---
**Cells/Workflow Order:**

| Cell # | Purpose | Expected Results |
|--------|---------|------------------|
| 1-2 | Setup (imports, random seed) | Environment configured |
| 3 | Temp output directory setup (nb_temp_setup) | Per-run temp dir, repo stays clean |
| 4 | Cleanup previous outputs | No stale data remains |
| 5 | Start ES container + credentials | Docker running, credentials file generated |
| 6-7 | Populate dummy patient data + ingest BMI | Documents in Elasticsearch |
| 8 | Index refresh verification | All indices have documents |
| 9 | Generate + ingest Epic medical history + count verification | epic_medical_history index populated |
| 10-11 | Initialize database and logger | SQLite DB created |
| 12-13 | Create pat2vec config with epic_medical_history_annotations mode | Config with correct options |
| 14-15 | Run pat2vec pipeline | Pipeline processes patients successfully |
| 16-18 | Extract all features from database | Features DataFrame populated |
| 19 | Feature preview | Sample of extracted features displayed |
| 20 | Data retrieval test (real batch from ES -> annotations -> non-empty count features) | Non-empty batch with positive count features |
| 21 | End-to-end verification (ingested data present in feature store) | Annotation features present with positive counts |
| 22-23 | Merge builder functionality | Merge function returns non-empty feature data |
| 24 | Cleanup database, project directory, ES container, temp dir | All temp files deleted |
| 25 | Final verification | All assertions pass, TEST SUCCESSFUL |

---
**Test Failure Conditions:**
- Any cell raises unhandled exception
- Elasticsearch container fails to start
- No patient IDs generated after population
- Empty DataFrame from feature extraction
- epic_medical_history index has fewer documents than expected after ingestion
- Retrieved annotation batch is empty
- No pretty_name_count_epic_medical_history_* features with positive counts

In [ ]:
import os
import random
import shutil
import sys

import numpy as np

random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
import os
import sys

_here = os.getcwd()
for _c in (_here, os.path.join(_here, "notebooks", "test")):
    if os.path.exists(os.path.join(_c, "nb_temp_setup.py")):
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break
else:
    raise FileNotFoundError(
        "nb_temp_setup.py not found in the working directory or "
        "notebooks/test. Run this notebook from notebooks/test or the "
        "repository root."
    )

from nb_temp_setup import cleanup_nb_temp_dir, setup_nb_temp_dir

# All relative-path artifacts (project dir, SQLite DB, credentials, logs)
# are now written to this per-run temp folder instead of the repository.
nb_temp_dir, repo_root = setup_nb_temp_dir()
print(f"Notebook output directory: {nb_temp_dir}")
print(f"Repository root: {repo_root}")

In [ ]:
for dir_to_remove in ["epic_medical_history_annotations_test_project"]:
    if os.path.exists(dir_to_remove):
        try:
            shutil.rmtree(dir_to_remove)
        except Exception as e:
            raise RuntimeError(
                f"Failed to clean up '{dir_to_remove}' directory: {e}. "
                "Critical error - cannot start with stale data."
            ) from e

print("Previous outputs cleaned.")

In [ ]:
from nb_temp_setup import get_notebook_port_offsetfrom pat2vec.util.docker_elastic import ElasticContainerport_offset = get_notebook_port_offset()es_container = ElasticContainer(port_offset=port_offset)es_container.stop()print("Starting Elasticsearch container (this may take a few seconds)...")if not es_container.start():    raise RuntimeError(        "Failed to start Elasticsearch container. Check if Docker is running."    )host, username, password = es_container.get_credentials()creds_filename = "test_elastic_credentials.py"creds_content = f"""username = "{username}"password = "{password}"api_key = Nonehosts = ["{host}"]"""with open(creds_filename, "w") as f:    f.write(creds_content)print(f"Created '{creds_filename}' pointing to test cluster at {host}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

schema_path = os.path.join(repo_root, "notebooks", "test_files", "elastic_schemas.json")
if not os.path.exists(schema_path):
    raise RuntimeError(f"Elasticsearch test schema not found: {schema_path}")
config_populate = config_class(
    proj_name="epic_medical_history_annotations_test_project",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print()
print("Population complete.")
print(f"Generated {len(patient_ids)} dummy patients.")
print(f"Patient IDs: {patient_ids}")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
print("Indices refreshed.")

print()
print("Index Status:")
for index in indices:
    try:
        if cs.elastic.indices.exists(index=index):
            count = cs.elastic.count(index=index)["count"]
            print(f"  - {index:<20}: {count} documents")
        else:
            raise RuntimeError(f"Index not created: {index}")
    except Exception as e:
        raise RuntimeError(f"Error checking index {index}: {e}")

In [ ]:
import pandas as pd

# Generate and ingest BMI data to Elasticsearch
from pat2vec.util.get_dummy_data_cohort_searcher import generate_bmi_data
from pat2vec.util.elasticsearch_methods import ingest_data_to_elasticsearch

# Generate BMI data for each patient
bmi_dfs = []
for pid in patient_ids:
    df = generate_bmi_data(
        num_rows=3,
        entered_list=[pid],
        global_start_year=int(config_populate.global_start_year),
        global_start_month=int(config_populate.global_start_month),
        global_end_year=int(config_populate.global_end_year),
        global_end_month=int(config_populate.global_end_month),
    )
    bmi_dfs.append(df)

# Combine all BMI data
df_bmi = pd.concat(bmi_dfs, ignore_index=True) if len(bmi_dfs) > 1 else bmi_dfs[0]
df_bmi = df_bmi.where(pd.notnull(df_bmi), None)

# Ingest into Elasticsearch
ingest_data_to_elasticsearch(df_bmi, "observations", es_client=cs.elastic)
cs.elastic.indices.refresh(index="observations")

print(f"Ingested {len(df_bmi)} BMI observations for {len(patient_ids)} patients")

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import (
    generate_epic_medical_history_data,
)

print("Generating and ingesting Epic medical history data...")
medical_history_dfs = []
for pid in patient_ids:
    df = generate_epic_medical_history_data(
        num_rows=3,
        entered_list=[pid],
        global_start_year=int(config_populate.global_start_year),
        global_start_month=int(config_populate.global_start_month),
        global_end_year=int(config_populate.global_end_year),
        global_end_month=int(config_populate.global_end_month),
    )
    medical_history_dfs.append(df)

df_medical_history = (
    pd.concat(medical_history_dfs, ignore_index=True)
    if len(medical_history_dfs) > 1
    else medical_history_dfs[0]
)
df_medical_history = df_medical_history.where(pd.notnull(df_medical_history), None)

if df_medical_history.empty:
    raise RuntimeError(
        "FATAL ERROR: generated Epic medical history DataFrame is empty. "
        "Dummy data generation failed."
    )

ingest_data_to_elasticsearch(
    df_medical_history,
    "epic_medical_history",
    es_client=cs.elastic,
)
cs.elastic.indices.refresh(index="epic_medical_history")

mh_count = cs.elastic.count(index="epic_medical_history")["count"]
expected_mh_count = len(patient_ids) * 3
if mh_count < expected_mh_count:
    raise RuntimeError(
        f"FATAL ERROR: epic_medical_history index has {mh_count} documents, "
        f"expected at least {expected_mh_count}. "
        "Ingestion into the test Elasticsearch cluster failed."
    )

print(
    f"Ingested {mh_count} Epic medical history documents "
    f"(expected >= {expected_mh_count}) into 'epic_medical_history'."
)

In [ ]:
PROJ_NAME = "epic_medical_history_annotations_test_project"
DB_FILENAME = "temp_bmi_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    raise RuntimeError(
        f"Failed to remove old database file '{DB_PATH}': {e}. "
        "Critical error - cannot start with stale data."
    ) from e

db_connection_string = f"sqlite:///{DB_PATH}"
print(f"Database connection string set to: {db_connection_string}")
print(f"(resolves to: {os.path.abspath(DB_PATH)})")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"epic_medical_history_annotations": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    # Match the pipeline's global extraction window to the window used
    # when the dummy Epic medical history data was ingested (cell 9),
    # so no ingested rows fall outside the ES fetch range.
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
    all_patient_list=patient_ids,
)

print(
    "pat2vec configuration created with epic_medical_history_annotations mode "
    "and database backend."
)

In [ ]:
from pat2vec.main_pat2vec import main

try:
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except FileNotFoundError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: config path invalid. Error details: {e}."
    ) from e
except ValueError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: invalid configuration. Error details: {e}."
    ) from e
except RuntimeError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: initialization failure. Error details: {e}."
    ) from e
except Exception as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: unexpected error. Error details: {e}."
    ) from e

print("pat2vec object initialized.")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

In [ ]:
if not pat2vec_obj.all_patient_list:
    raise RuntimeError(
        "No patients in patient list after initialization. "
        "This indicates a critical failure in data loading or filtering."
    )

print(f"Processing patient: {pat2vec_obj.all_patient_list[0]}")

try:
    pat2vec_obj.pat_maker(0)
except Exception as e:
    raise RuntimeError(
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    raise RuntimeError(
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database."
    )

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
all_features_alt = pat2vec_obj.get_all_features()

if all_features_alt.empty:
    raise RuntimeError(
        "FATAL ERROR: pat2vec_obj.get_all_features() returned an empty DataFrame. "
        "This indicates a critical failure in feature storage."
    )

print(f"pat2vec_obj.get_all_features(): {all_features_alt.shape[0]} rows retrieved.")

In [ ]:
from pat2vec.util.post_processing import extract_datetime_to_column

df_with_datetime = extract_datetime_to_column(all_features)

print()
print("=== OUTPUT FEATURES DATAFRAME ===")
print(f"Shape: {df_with_datetime.shape}")
print(f"Total features: {len(df_with_datetime.columns)}")

if not df_with_datetime.empty:
    print()
    print("First 3 rows:")
    print(df_with_datetime.head(3))
else:
    raise RuntimeError(
        "DataFrame is empty after datetime extraction. Critical error - no features to extract."
    )

In [ ]:
print(
    "\n=== DEMONSTRATING DATA RETRIEVAL FOR EPIC_MEDICAL_HISTORY_ANNOTATIONS MODE ==="
)

all_pat_list = pat2vec_obj.all_patient_list
current_pat = all_pat_list[0]

from pat2vec.patvec_get_batch_methods.main_get_pat_batch_epic_medical_history_annotations import (
    get_pat_batch_epic_medical_history_annotations,
)
from pat2vec.pat2vec_get_methods.get_method_epic_medical_history_annotations import (
    get_current_pat_epic_medical_history_annotations,
)

# Fetch the patient's real batch: raw epic_medical_history documents from the
# test Elasticsearch cluster, annotated with the dummy MedCAT model.
pat_batch = get_pat_batch_epic_medical_history_annotations(
    current_pat_client_id_code=current_pat,
    config_obj=config_obj,
    cat=pat2vec_obj.cat,
    t=None,
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
)

if pat_batch is None or pat_batch.empty:
    raise RuntimeError(
        "FATAL ERROR: get_pat_batch_epic_medical_history_annotations returned an empty batch. "
        "The Epic medical history data ingested into the test Elasticsearch cluster "
        "was not retrieved. Check the 'epic_medical_history' index population."
    )

print(
    f"Retrieved {len(pat_batch)} annotated medical history rows for patient {current_pat}"
)
print(f"Batch columns: {list(pat_batch.columns)}")

# Provenance check: the retrieved documents must be the ones we ingested into
# Elasticsearch. This prevents the test from passing on a silent dummy-data
# fallback (e.g. if the ES fetch returned nothing and the pipeline generated a
# synthetic 'dummy_doc_*' batch instead of the real ingested documents).
ingested_ids = set(
    df_medical_history.loc[
        df_medical_history["document_PatientDurableKey"] == current_pat, "id"
    ].astype(str)
)
if "document_guid" not in pat_batch.columns:
    raise RuntimeError(
        "FATAL ERROR: retrieved batch has no 'document_guid' column, so its "
        f"provenance cannot be verified. Columns: {list(pat_batch.columns)}"
    )
batch_guids = set(pat_batch["document_guid"].astype(str))
unknown_guids = batch_guids - ingested_ids
if not batch_guids or unknown_guids:
    raise RuntimeError(
        f"FATAL ERROR: retrieved batch document GUIDs {sorted(batch_guids)[:5]} are "
        f"not the documents ingested into Elasticsearch for patient {current_pat} "
        f"(ingested: {sorted(ingested_ids)[:5]}). The pipeline did not extract the "
        "ingested Epic medical history data (likely a silent dummy-data fallback)."
    )
print(
    f"Verified {len(batch_guids & ingested_ids)}/{len(ingested_ids)} retrieved "
    "document GUIDs match the ingested Epic medical history documents."
)

epic_mh_ann_data = get_current_pat_epic_medical_history_annotations(
    current_pat_client_id_code=current_pat,
    target_date_range=(2020, 1, 1, 2023, 12, 31),
    epic_medical_history_annotations=pat_batch,
    config_obj=config_obj,
)

if epic_mh_ann_data is None or epic_mh_ann_data.empty:
    raise RuntimeError(
        "FATAL ERROR: get_current_pat_epic_medical_history_annotations returned empty result. "
        "This indicates a critical failure in epic_medical_history_annotations feature extraction."
    )

feature_cols = [
    c
    for c in epic_mh_ann_data.columns
    if c.startswith("pretty_name_count_epic_medical_history_")
]
if not feature_cols:
    raise RuntimeError(
        "FATAL ERROR: no 'pretty_name_count_epic_medical_history_*' feature columns in "
        f"extracted data. Columns present: {list(epic_mh_ann_data.columns)}. "
        "The ingested medical history annotations were not converted to features."
    )

feature_values = epic_mh_ann_data[feature_cols].fillna(0).to_numpy().astype(float)
if feature_values.sum() <= 0:
    raise RuntimeError(
        "FATAL ERROR: all epic_medical_history annotation feature values are zero. "
        "No annotation counts were calculated from the ingested data."
    )

print(f"\nepic_medical_history_annotations feature columns: {feature_cols}")
print("\nSample features:")
print(epic_mh_ann_data)

In [ ]:
# === VECTOR VALIDATION ===
feature_cols = [c for c in all_features.columns if c.startswith("med_hist_")]

assert (
    len(feature_cols) > 0
), f"No feature columns found. Available columns: {list(all_features.columns)}"

non_null_counts = all_features[feature_cols].notna().sum()
totally_empty = non_null_counts[non_null_counts == 0]

assert len(totally_empty) == 0, (
    f"The following feature columns are entirely null:\n"
    f"{list(totally_empty.index)}\n"
    "Vectorisation is silently failing — check the get method return value."
)

feature_values = all_features[feature_cols].fillna(0).to_numpy().astype(float)
if feature_values.sum() <= 0:
    raise RuntimeError(
        "FATAL ERROR: all feature values are zero or negative. "
        "Feature extraction produced no meaningful numeric output."
    )

print("Feature columns (" + str(len(feature_cols)) + "): " + str(feature_cols))
print("Non-null counts per feature column:")
for col in sorted(feature_cols):
    val = all_features[col].notna().sum()
    print("  " + str(col) + ": " + str(val) + " non-null values")

In [ ]:
print("\n=== VERIFYING INGESTED DATA APPEARS IN EXTRACTED FEATURES ===")

annotation_feature_cols = [
    c
    for c in all_features.columns
    if c.startswith("pretty_name_count_epic_medical_history_")
]

if not annotation_feature_cols:
    raise RuntimeError(
        "FATAL ERROR: no epic_medical_history annotation feature columns present in the "
        f"extracted features. Columns: {list(all_features.columns)[:20]}... "
        "The Epic medical history data ingested into Elasticsearch did not flow "
        "through the pat2vec pipeline."
    )

annotation_values = (
    all_features[annotation_feature_cols].fillna(0).to_numpy().astype(float)
)
if annotation_values.sum() <= 0:
    raise RuntimeError(
        "FATAL ERROR: all epic_medical_history annotation feature values in the "
        "feature store are zero. Ingested data was not extracted."
    )

print(f"Verified {len(annotation_feature_cols)} annotation feature columns:")
print(f"  {annotation_feature_cols}")
print(f"Total annotation counts in feature store: {int(annotation_values.sum())}")

In [ ]:
import os
import pandas as pd

from pat2vec.util.helper_functions import get_all_features

from pat2vec.util.post_processing_build_methods import (
    build_merged_epr_mct_annot_df,
)

print("\n=== DEMONSTRATING FEATURE MERGE FUNCTIONALITY ===")

merged_path = build_merged_epr_mct_annot_df(
    all_pat_list,
    config_obj,
    overwrite=True,
)

assert (
    merged_path is not None
), "build_merged_epr_mct_annot_df returned None — expected a file path"
assert os.path.exists(merged_path), f"Merged file does not exist at {merged_path}"

csv_data = pd.read_csv(merged_path)
assert not csv_data.empty, (
    "Merged DataFrame is empty after pat_maker ran. "
    "pat_maker likely encountered a table error — check output above for "
    "'no such table' or 'OperationalError' messages."
)

print(f"Merged data saved to: {merged_path}")
print(f"Shape: {csv_data.shape}")

table_error_strings = [
    "no such table",
    "operationalerror",
    "table not found",
    "no table named",
]
print("=== TABLE ERROR CHECK ===")
print(
    "If pat_maker output above contains any of these strings, "
    "the merge builders will fail:"
)
for s in table_error_strings:
    print(f"  - '{s}'")
print("Proceeding to merge builders...")

In [ ]:
print("\n=== DATABASE AND PROJECT CLEANUP ===")

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove database file '{DB_PATH}': {e}. Critical error - cleanup incomplete."
    ) from e

try:
    if os.path.exists(PROJ_NAME):
        shutil.rmtree(PROJ_NAME, ignore_errors=False)
        print(f"Removed project directory: {PROJ_NAME}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove '{PROJ_NAME}' directory: {e}. Critical error - cleanup incomplete."
    ) from e

try:
    if os.path.exists(creds_filename):
        os.remove(creds_filename)
        print(f"Removed Elasticsearch credentials: {creds_filename}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove Elasticsearch credentials file '{creds_filename}': {e}. "
        "Critical error - cleanup incomplete."
    ) from e

try:
    es_container.stop()
    print("Stopped Elasticsearch test container.")
except Exception as e:
    print(f"Warning: could not stop Elasticsearch container: {e}")

cleanup_nb_temp_dir(nb_temp_dir)
print(f"Removed temp output directory: {nb_temp_dir}")

In [ ]:
print("\n=== FINAL VERIFICATION ===")

assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(PROJ_NAME), "Project directory still exists!"
assert not os.path.exists(
    creds_filename
), "Elasticsearch credentials file still exists!"

leftovers = [
    e
    for e in os.listdir(os.path.dirname(nb_temp_dir))
    if e.startswith("pat2vec_nb_")
    and os.path.join(os.path.dirname(nb_temp_dir), e) == nb_temp_dir
]
assert not leftovers, f"Temp output directory still exists: {nb_temp_dir}"

print("All cleanup verified - temp output directory fully removed.")
print("\n=== TEST SUCCESSFUL ===")